# Part 5 – Object Matching & Change Events

**Branch:** `feature/object-matching-events`


## 11. Detect Objects, Align, Build Pixel-Change Maps

### 11a. One pass over all pairs (YOLO at conf 0.10, both alignments, change map, appearance histograms)

In [ ]:
CHANGE_LABEL_MAP = {'add': 'Add', 'delete': 'Delete', 'move': 'Move'}

pairs_df = ALL_PAIRS_DF.copy()
pairs_df['true_change'] = pairs_df['change_type'].map(CHANGE_LABEL_MAP)
assert pairs_df['true_change'].notna().all(), "Unexpected change_type in ALL_PAIRS_DF"
pairs_df = pairs_df.rename(columns={'object': 'true_object'})

print(f"{len(pairs_df)} labeled reference/query pairs")
pairs_df[['pair_id', 'reference_path', 'query_path', 'true_change', 'true_object', 'room', 'split']].head()

In [ ]:
def raw_version(path):
    p = os.path.join(ORIG_BACKUP_DIR, os.path.basename(path))
    return p if os.path.exists(p) else path

def prepare_pair(reference_boxes, query_boxes, ref_pre, qry_pre, ref_raw, qry_raw):
    attach_histograms(reference_boxes, ref_pre)
    attach_histograms(query_boxes, qry_pre)
    H, residual_px = estimate_camera_shift(ref_pre, qry_pre)
    H_v40, residual_px_v40 = estimate_camera_shift_v40(ref_pre, qry_pre)
    qh, qw = qry_pre.shape[:2]
    return {'reference_boxes': reference_boxes, 'query_boxes': query_boxes,
            'H': H, 'residual_px': residual_px, 'H_v40': H_v40, 'residual_px_v40': residual_px_v40,
            'image_diag': float(np.hypot(qw, qh)),
            'change': compute_change_map(ref_raw, qry_raw, H)}

detection_cache = []
BATCH_SIZE = 16
pair_rows = pairs_df.to_dict('records')

for batch_start in range(0, len(pair_rows), BATCH_SIZE):
    batch = pair_rows[batch_start: batch_start + BATCH_SIZE]
    batch_paths = [p for row in batch for p in (row['reference_path'], row['query_path'])]
    batch_results = yolo_model(batch_paths, conf=LOW_CONF_THRESH, verbose=False)

    for k, row in enumerate(batch):
        entry = prepare_pair(
            extract_boxes([batch_results[2 * k]], yolo_model), extract_boxes([batch_results[2 * k + 1]], yolo_model),
            cv2.imread(row['reference_path']), cv2.imread(row['query_path']),
            cv2.imread(raw_version(row['reference_path'])), cv2.imread(raw_version(row['query_path'])))
        entry.update({
            'pair_id': row['pair_id'], 'reference_path': row['reference_path'], 'query_path': row['query_path'],
            'true_change': row['true_change'], 'true_object': row['true_object'], 'room': row['room'],
            'split': row['split'],
        })
        detection_cache.append(entry)
    if (batch_start // BATCH_SIZE) % 10 == 0:
        print(f"  processed {min(batch_start + BATCH_SIZE, len(pair_rows))}/{len(pair_rows)} pairs")

n_strong = sum(b['confidence'] >= CONF_THRESH for c in detection_cache for b in c['reference_boxes'] + c['query_boxes'])
n_all = sum(len(c['reference_boxes']) + len(c['query_boxes']) for c in detection_cache)
n_fallback_H = sum(np.allclose(c['H'], np.eye(3)) and c['residual_px'] == 0.0 for c in detection_cache)
print(f"Detections, alignment and pixel-change maps ready for {len(detection_cache)} pairs: "
      f"{n_strong} confident boxes (>= {CONF_THRESH}) + {n_all - n_strong} weak evidence boxes "
      f"(>= {LOW_CONF_THRESH}); {n_fallback_H} pairs fell back to identity alignment.")


demo_entry = prepare_pair(
    extract_boxes(yolo_model(demo_reference_pre, conf=LOW_CONF_THRESH, verbose=False), yolo_model),
    extract_boxes(yolo_model(demo_query_pre, conf=LOW_CONF_THRESH, verbose=False), yolo_model),
    demo_reference_pre, demo_query_pre, demo_reference_bgr, demo_query_bgr)
demo_entry.update({'pair_id': 'demo', 'reference_path': USER_REFERENCE_PATH, 'query_path': USER_QUERY_PATH,
                   'true_change': None, 'true_object': None, 'room': None, 'split': 'demo'})
print(f"Demo pair prepared: {len(demo_entry['reference_boxes'])} reference / {len(demo_entry['query_boxes'])} "
      f"query boxes (all confidences >= {LOW_CONF_THRESH}).")

### 11b. Tune the rule thresholds on the VALIDATION split


In [ ]:
from itertools import product

val_cache = [c for c in detection_cache if c['split'] == 'validation']
test_cache = [c for c in detection_cache if c['split'] == 'test']


TUNING_GRID = {
    'pixel_k':        [2.5, 3.0, 4.0],
    'visual_sat':     [0.20, 0.35, 0.50],
    'min_box_change': [0.05, 0.10, 0.20],
    'move_rel_frac':  [0.10, 0.20, 0.30],
    'ghost_iou':      [0.30, 0.50],
}

tuning_rows = []
for combo in product(*TUNING_GRID.values()):
    trial = {**DEFAULT_PARAMS, **dict(zip(TUNING_GRID, combo))}
    _, trial_summary, _ = build_events_df(val_cache, trial)
    tuning_rows.append({**dict(zip(TUNING_GRID, combo)), **pair_metrics(trial_summary)})

tuning_df = (pd.DataFrame(tuning_rows)
             .sort_values(['accuracy', 'macro_f1', 'type_and_object_acc'], ascending=False, kind='stable')
             .reset_index(drop=True))
best_row = tuning_df.iloc[0]
FINAL_PARAMS = {**DEFAULT_PARAMS, **{k: float(best_row[k]) for k in TUNING_GRID}}

print(f"Searched {len(tuning_df)} threshold combinations on the {len(val_cache)} VALIDATION pairs.")
print("Top 10 (validation):")
display(tuning_df.head(10).round(3))
print("Chosen thresholds:", {k: FINAL_PARAMS[k] for k in TUNING_GRID})


fig, axes = plt.subplots(1, len(TUNING_GRID), figsize=(3.2 * len(TUNING_GRID), 3.2), sharey=True)
for ax, key in zip(axes, TUNING_GRID):
    g = tuning_df.groupby(key)['accuracy'].agg(['mean', 'max'])
    ax.plot(g.index, g['mean'], 'o-', label='mean over grid')
    ax.plot(g.index, g['max'], 's--', label='best')
    ax.set_title(key, fontsize=10); ax.grid(alpha=0.3)
axes[0].set_ylabel('Validation accuracy'); axes[0].legend(fontsize=8)
plt.suptitle('Threshold sensitivity (VALIDATION split)')
plt.tight_layout()
plt.savefig('/content/threshold_sensitivity_val.png', dpi=150)
plt.show()

### 11c. Final run with the tuned thresholds

In [ ]:
events_df, pairs_summary_df, pair_contexts = build_events_df(detection_cache, FINAL_PARAMS)

print(f"{len(detection_cache)} pairs -> {len(events_df)} candidate object-level change events "
      f"({int(events_df['is_final_change'].sum())} pass the {FINAL_PARAMS['prob_threshold']*100:.0f}% "
      f"final-decision threshold).")
print(f"Pairs decided by the low-confidence fallback pass: "
      f"{(pairs_summary_df['decision_pass'] != 'strict').sum()}")

# ---- demo pair ----
demo_result = detect_changes(demo_entry, FINAL_PARAMS)
demo_reference_boxes, demo_query_boxes = demo_result['reference_boxes'], demo_result['query_boxes']
demo_H, demo_residual_px = demo_result['H'], demo_result['residual_px']
demo_move_thresh = demo_result['move_thresh']
demo_image_diag = demo_entry['image_diag']
demo_events, demo_matched_pairs = demo_result['events'], demo_result['matched_pairs']
demo_suppressed, demo_decision = demo_result['suppressed'], demo_result['decision']
demo_stats = summarize_events(demo_events, demo_matched_pairs)
demo_ctx = {'stats': demo_stats, 'matched_pairs': demo_matched_pairs, 'events': demo_events,
            'suppressed': demo_suppressed, 'decision': demo_decision, 'pass': demo_result['pass']}

print(f"\nDemo pair: {len(demo_reference_boxes)} reference box(es), {len(demo_query_boxes)} query box(es), "
      f"{len(demo_matched_pairs)} matched, {len(demo_events)} candidate change event(s), "
      f"{len(demo_suppressed)} cross-class pair(s) suppressed as detector noise  [{demo_result['pass']} pass].")
print(f"FINAL DECISION -- {demo_decision['summary']}")

## 12. Matching Algorithm — Input and Output

**Input:** YOLO boxes (class, bbox, confidence) for reference and query -- confident boxes create events, weak ones
(0.10–0.40) are evidence only -- the homography `H` (reference → query), the thresholds, the appearance histograms
and the pixel change map.
**Output:** `matched_pairs` (with a role: unchanged / Move / one-sided Move turned into Add or Delete) and `events` —
the per-object Add / Delete / Move labels, each with its change-probability and visual-change fraction.

In [ ]:
pos_by_pair_id = {c['pair_id']: i for i, c in enumerate(detection_cache)}

def move_thresh_for(c):
    return adaptive_move_thresh(c['residual_px'], MOVE_MIN_THRESH, MOVE_SAFETY_FACTOR)

def boxes_df(boxes):
    return pd.DataFrame(
        [{'class': b['class'], 'x1': round(b['bbox'][0]), 'y1': round(b['bbox'][1]),
          'x2': round(b['bbox'][2]), 'y2': round(b['bbox'][3]),
          'confidence': round(b['confidence'], 3)} for b in boxes],
        columns=['class', 'x1', 'y1', 'x2', 'y2', 'confidence'])

def matching_output_tables(ctx, move_thresh=None):
    matched_df = pd.DataFrame(
        [{'class': b['class'],
          'reference_bbox': [round(v) for v in b['bbox']],
          'query_bbox': [round(v) for v in a['bbox']],
          'moved_px': round(dist, 1),
          'appearance_sim': round(sim, 2),
          'role': role}
         for b, a, dist, sim, role in ctx['matched_pairs']],
        columns=['class', 'reference_bbox', 'query_bbox', 'moved_px', 'appearance_sim', 'role'])

    final_ids = {id(e) for e in ctx['decision']['changes']} if 'decision' in ctx else set()
    events_df = pd.DataFrame(
        [{'change_type': e['type'], 'object': e['object'],
          'probability_pct': round(e['probability'] * 100, 1),
          'confidence': round(e['confidence'], 3), 'movement_px': round(e['movement'], 1),
          'visual_change_pct': None if e.get('visual_change') is None else round(e['visual_change'] * 100, 1),
          'ghost': e.get('ghost', False),
          'final_decision': 'YES' if id(e) in final_ids else 'no (below threshold)',
          'note': e.get('note', '')}
         for e in sorted(ctx['events'], key=lambda e: -e['probability'])],
        columns=['change_type', 'object', 'probability_pct', 'confidence', 'movement_px',
                 'visual_change_pct', 'ghost', 'final_decision', 'note'])

    suppressed_df = pd.DataFrame(ctx.get('suppressed', []),
        columns=['ref_class', 'qry_class', 'iou', 'note'])

    return matched_df, events_df, suppressed_df

def annotate_side(img_bgr, boxes, matched_ids, move_ids, side):
    out = img_bgr.copy()
    for b in boxes:
        x1, y1, x2, y2 = b['bbox']
        if id(b) in move_ids:
            draw_labeled_box(out, x1, y1, x2, y2, f"{b['class']} - Move", (0, 140, 255))
        elif id(b) in matched_ids:
            draw_labeled_box(out, x1, y1, x2, y2, b['class'], (160, 160, 160), box_thickness=2)
        elif side == 'reference':
            draw_labeled_box(out, x1, y1, x2, y2, f"{b['class']} - Delete", (0, 0, 255))
        else:
            draw_labeled_box(out, x1, y1, x2, y2, f"{b['class']} - Add", (0, 180, 0))
    return out

### 12a. Matching input

In [ ]:
print("Matching input -- demo pair")
print(f"H (reference -> query):\n{np.array2string(demo_H, precision=4, suppress_small=True)}")
print(f"camera_shift_residual = {demo_residual_px:.2f}px  |  base move_thresh = {demo_move_thresh:.1f}px  |  "
      f"image_diag = {demo_image_diag:.1f}px")
print(f"max_match_dist_frac = {FINAL_PARAMS['max_match_dist_frac']}  |  min_appearance_sim = {FINAL_PARAMS['min_appearance_sim']}  |  "
      f"recovery_appearance_sim = {FINAL_PARAMS['recovery_sim']}  |  cross_class_suppress_iou = {FINAL_PARAMS['cross_class_suppress_iou']}")
print(f"cost weights -- w_dist={FINAL_PARAMS['w_dist']}  w_iou={FINAL_PARAMS['w_iou']}  w_app={FINAL_PARAMS['w_app']}")
print(f"pixel evidence -- pixel_k={FINAL_PARAMS['pixel_k']}  visual_sat={FINAL_PARAMS['visual_sat']}  "
      f"min_box_change={FINAL_PARAMS['min_box_change']}  ghost_iou={FINAL_PARAMS['ghost_iou']}  "
      f"move_rel_frac={FINAL_PARAMS['move_rel_frac']}")

print(f"\nReference boxes ({len(demo_reference_boxes)} confident):")
display(boxes_df(demo_reference_boxes))
print(f"Query boxes ({len(demo_query_boxes)} confident):")
display(boxes_df(demo_query_boxes))

### 12b. Matching output

In [ ]:
matched_df, candidate_events_df, suppressed_df = matching_output_tables(demo_ctx)

print(f"Matching output -- demo pair: {len(matched_df)} matched pair(s), {len(demo_events)} candidate event(s), "
      f"{len(demo_suppressed)} cross-class pair(s) suppressed as detector noise.")
print("\nMatched pairs (per-class Hungarian assignment, incl. recovered / merged same-object Moves):")
display(matched_df)

print("\nCandidate change events, ranked by change-probability (this is BEFORE the final decision filter):")
display(candidate_events_df)

if len(suppressed_df):
    print("\nCross-class pairs suppressed as likely detector class-flip noise (not reported as Add+Delete):")
    display(suppressed_df)

## 13. Per-Object Change Events (Add / Delete / Move) -- with change-probability & final decision



### 13a. Events across the dataset

In [ ]:
print(f"{len(events_df)} total CANDIDATE change events across {len(detection_cache)} pairs")
print(f"{int(events_df['is_final_change'].sum())} events pass the {FINAL_PARAMS['prob_threshold']*100:.0f}% "
      f"final-decision threshold and are actually reported as changes")
print()
print("Candidate events by type:")
print(events_df['change_type'].value_counts())
print("\nFinal (reported) events by type:")
print(events_df[events_df['is_final_change']]['change_type'].value_counts())

print(f"\nPairs with more than one CANDIDATE event: {(pairs_summary_df['n_events'] > 1).sum()} / {len(pairs_summary_df)}")
print(f"Pairs with more than one FINAL (reported) change: {(pairs_summary_df['n_final_changes'] > 1).sum()} / {len(pairs_summary_df)}")
print(f"Pairs with at least one cross-class pair suppressed as detector noise: "
      f"{(pairs_summary_df['n_suppressed_cross_class'] > 0).sum()} / {len(pairs_summary_df)}")
print(f"Pairs decided by the low-confidence fallback pass: "
      f"{(pairs_summary_df['decision_pass'] != 'strict').sum()} / {len(pairs_summary_df)}")

events_df.groupby('split')['change_type'].value_counts().unstack(fill_value=0)

### 13b. Demo pair -- full multi-object event list, change-probability & final decision


In [ ]:
demo_events_df = pd.DataFrame(
    [{'object': e['object'], 'change_type': e['type'],
      'probability_pct': round(e['probability'] * 100, 1),
      'confidence': round(e['confidence'], 3), 'movement_px': round(e['movement'], 1),
      'visual_change_pct': None if e.get('visual_change') is None else round(e['visual_change'] * 100, 1),
      'final_decision': 'YES' if any(e is c for c in demo_decision['changes']) else 'no (below threshold)'}
     for e in sorted(demo_events, key=lambda e: -e['probability'])],
    columns=['object', 'change_type', 'probability_pct', 'confidence', 'movement_px', 'visual_change_pct',
             'final_decision'])

print(f"--- Demo pair: {len(demo_events_df)} candidate change event(s), "
      f"{len(demo_decision['changes'])} pass the {FINAL_PARAMS['prob_threshold']*100:.0f}% threshold "
      f"[{demo_ctx['pass']} pass] ---")
display(demo_events_df)

print(f"\nFINAL DECISION: {demo_decision['summary']}")